In [1]:
!pip install mamba-ssm causal-conv1d --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 3.7 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 38.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 39.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 69.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 18.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.4/772.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 55.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 79.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4

# Prerequisites

In [3]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from transformers import AutoConfig, AutoModelForImageClassification, AutoModel, AutoTokenizer
from tqdm import tqdm

# --- Configuration ---
IMG_DIR = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017/"
ANN_FILE = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_train2017.json"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
VISION_MODEL = "saurabhati/VMamba_ImageNet_82.6"
TEXT_MODEL = "state-spaces/mamba-130m-hf"

BATCH_SIZE = 64
LATENT_DIM = 512
LR = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Dataset Loader

In [4]:
class COCODataset(Dataset):
    def __init__(self, img_dir, ann_file, tokenizer):
        self.img_dir = img_dir
        with open(ann_file, 'r') as f:
            data = json.load(f)
            
        # Create a mapping of image_id to filename
        self.img_map = {img['id']: img['file_name'] for img in data['images']}
        self.annotations = data['annotations']
        
        self.preprocess = T.Compose([
            T.Resize(224, interpolation=Image.BICUBIC),
            T.CenterCrop(224),
            T.ToTensor(),
            T.Normalize(mean=[0.4850, 0.4560, 0.4060], std=[0.2290, 0.2240, 0.2250])
        ])
        
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_filename = self.img_map[ann['image_id']]
        img_path = os.path.join(self.img_dir, img_filename)
        
        image = Image.open(img_path).convert('RGB')
        image = self.preprocess(image)
        
        text = ann['caption']
        text_inputs = self.tokenizer(
            text, 
            padding='max_length', 
            truncation=True, 
            max_length=64, 
            return_tensors="pt"
        )
        
        return {
            'image': image,
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0)
        }

# CLIMP model

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForImageClassification, AutoModel

# --- PATCH 1: Link mamba-ssm 2.x CUDA kernels for Hugging Face ---
import mamba_ssm
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn, mamba_inner_fn
    from mamba_ssm.ops.triton.selective_state_update import selective_state_update
    
    # Attach them to the top-level module where HF looks for them
    mamba_ssm.selective_scan_fn = selective_scan_fn
    mamba_ssm.mamba_inner_fn = mamba_inner_fn
    mamba_ssm.selective_state_update = selective_state_update
    print("✅ Successfully linked mamba-ssm CUDA kernels!")
except ImportError as e:
    print(f"⚠️ Warning: Could not link CUDA kernels. Error: {e}")
# ----------------------------------------------------------------

class CLIMP(nn.Module):
    def __init__(self, vision_model_name, text_model_name, latent_dim):
        super().__init__()
        
        # --- PATCH 2: Force torch.linspace to CPU to avoid meta-tensor crash in VMamba ---
        _orig_linspace = torch.linspace
        def _cpu_linspace(*args, **kwargs):
            if kwargs.get("device") is None:
                kwargs["device"] = torch.device("cpu")
            return _orig_linspace(*args, **kwargs)
        
        torch.linspace = _cpu_linspace
        # -------------------------------------------------------------------------------
        
        try:
            # 1. Vision Backbone (Frozen)
            self.vision_backbone = AutoModelForImageClassification.from_pretrained(
                vision_model_name, 
                trust_remote_code=True,
                low_cpu_mem_usage=False,
                _fast_init=False
            )
        finally:
            # Restore original linspace immediately after loading
            torch.linspace = _orig_linspace
            
        # Hack to get raw features instead of class logits
        if hasattr(self.vision_backbone, 'head'):
            vision_feat_dim = self.vision_backbone.head.in_features
            self.vision_backbone.head = nn.Identity()
        elif hasattr(self.vision_backbone, 'classifier'):
            vision_feat_dim = self.vision_backbone.classifier.in_features
            self.vision_backbone.classifier = nn.Identity()
        else:
            vision_feat_dim = self.vision_backbone.config.hidden_size 
            
        # 2. Text Backbone (Frozen)
        self.text_backbone = AutoModel.from_pretrained(
            text_model_name,
            trust_remote_code=True,
            low_cpu_mem_usage=False, 
            _fast_init=False
        )
        text_feat_dim = self.text_backbone.config.hidden_size
        
        # Freeze both backbones to save GPU memory 
        for param in self.vision_backbone.parameters():
            param.requires_grad = False
        for param in self.text_backbone.parameters():
            param.requires_grad = False
            
        # 3. Projection Heads (Trainable)
        self.vision_proj = nn.Linear(vision_feat_dim, latent_dim)
        self.text_proj = nn.Linear(text_feat_dim, latent_dim)
        
        # Learnable temperature parameter for InfoNCE loss
        self.logit_scale = nn.Parameter(torch.ones([]) * 2.6592) 
        
    def forward(self, images, input_ids, attention_mask):
        # --- Vision Features ---
        with torch.no_grad():
            v_out = self.vision_backbone(images)
            v_features = v_out.logits if hasattr(v_out, 'logits') else v_out 
            
        # --- Text Features (Mean Pooling) ---
        with torch.no_grad():
            t_out = self.text_backbone(input_ids=input_ids, attention_mask=attention_mask)
            t_hidden = t_out.last_hidden_state if hasattr(t_out, 'last_hidden_state') else t_out[0]
            
            mask_expanded = attention_mask.unsqueeze(-1).expand(t_hidden.size()).float()
            t_features = torch.sum(t_hidden * mask_expanded, 1) / torch.clamp(mask_expanded.sum(1), min=1e-9)

        # --- Project to Shared Space & L2 Normalize ---
        image_embeds = F.normalize(self.vision_proj(v_features), p=2, dim=-1)
        text_embeds = F.normalize(self.text_proj(t_features), p=2, dim=-1)
        
        return image_embeds, text_embeds, self.logit_scale.exp()

✅ Successfully linked mamba-ssm CUDA kernels!


# InfoNCE Loss & Training Loop

In [6]:
def info_nce_loss(image_embeds, text_embeds, logit_scale):
    # Cosine similarity scaled by temperature
    logits_per_image = logit_scale * image_embeds @ text_embeds.t()
    logits_per_text = logits_per_image.t()
    
    # The true pairs are on the diagonal
    labels = torch.arange(len(logits_per_image), device=logits_per_image.device)
    
    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    
    return (loss_i + loss_t) / 2

# Initialize Tokenizer (Mamba uses gpt-neox tokenizer which lacks a pad token)
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Data and Model
dataset = COCODataset(IMG_DIR, ANN_FILE, tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, drop_last=True)

model = CLIMP(VISION_MODEL, TEXT_MODEL, LATENT_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], # Only train projectors
    lr=LR, 
    weight_decay=0.01
)

# Training execution
model.train()
step = 0

print("Starting CLIMP training...")
for epoch in range(5): # Adjust epochs as needed
    for batch in tqdm(dataloader):
        images = batch['image'].to(DEVICE)
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        
        optimizer.zero_grad()
        
        image_embeds, text_embeds, logit_scale = model(images, input_ids, attention_mask)
        loss = info_nce_loss(image_embeds, text_embeds, logit_scale)
        
        loss.backward()
        optimizer.step()
        
        step += 1
        
        # Save complete model checkpoint every 1000 steps
        if step % 1000 == 0:
            chkpt_path = os.path.join(CHECKPOINT_DIR, f"climp_step_{step}.pt")
            torch.save({
                'step': step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': loss.item(),
            }, chkpt_path)
            print(f"Saved checkpoint: {chkpt_path} | Loss: {loss.item():.4f}")

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_vmamba.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/saurabhati/VMamba_ImageNet_82.6:
- configuration_vmamba.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


... (9 outputs left out)

 22%|██▏       | 2000/9246 [39:15<3:39:02,  1.81s/it]

Saved checkpoint: /kaggle/working/checkpoints/climp_step_2000.pt | Loss: 1.0594


 27%|██▋       | 2495/9246 [48:56<2:12:24,  1.18s/it]


KeyboardInterrupt: 